# Safe Qwen3.5 card enrichment notebook

This notebook is designed to avoid the repeated vLLM / FlashInfer MoE startup crash seen on Colab SM 12.x GPUs.

Default engine: **Transformers safe mode**.

It is slower than vLLM, but it avoids the failing `FlashInfer CUTLASS Unquantized MoE` path. The notebook also captures failed rows into a separate JSONL file so the full run does not stop because of bad generations or parsing issues.


In [ ]:
# 0. Environment preflight

import os
import sys
import subprocess
from pathlib import Path

def run_cmd(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout[-4000:])
    return p.returncode, p.stdout

print("Python:", sys.version)

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"GPU {i}: {props.name}, capability={props.major}.{props.minor}, total_vram={props.total_memory/1024**3:.1f} GB")
except Exception as e:
    print("Torch preflight failed, but continuing:", repr(e))

# Remove unsupported / noisy vLLM env vars from older attempts.
# This notebook does not use vLLM by default.
for k in [
    "VLLM_ATTENTION_BACKEND",
    "VLLM_MOE_BACKEND",
    "VLLM_FLASHINFER_MOE_BACKEND",
    "VLLM_WORKER_MULTIPROC_METHOD",
]:
    os.environ.pop(k, None)


In [ ]:
# 1. Install dependencies

# This uses Transformers instead of vLLM to avoid the FlashInfer MoE crash.
# Restart runtime only if Colab asks you to after package installation.

!pip install -q --upgrade transformers accelerate safetensors jsonschema tqdm

print("Dependency installation cell completed.")


In [ ]:
# 2. Config

from pathlib import Path

# Update this path to your actual input JSONL file.
# The input should contain one JSON object per line.
INPUT_FILE = Path("/content/court_authority_cards.jsonl")

OUT_DIR = Path("/content/drive/MyDrive/qwen35_card_enrichment_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUT_DIR / "enriched_cards.jsonl"
FAILED_FILE = OUT_DIR / "failed_cards.jsonl"
FAILED_RETRY_FILE = OUT_DIR / "failed_cards_retry_output.jsonl"
FAILED_STILL_FILE = OUT_DIR / "failed_cards_still_failed.jsonl"
CHECKPOINT_FILE = OUT_DIR / "checkpoint_processed_line_idxs.json"

MODEL_ID = "Qwen/Qwen3.5-35B-A3B"

# Safe default. This avoids vLLM entirely.
ENGINE = "transformers_safe"

# Conservative defaults to reduce OOM risk.
# Increase BATCH_SIZE slowly only after smoke test passes.
BATCH_SIZE = 1
MAX_NEW_TOKENS = 256
RETRY_MAX_NEW_TOKENS = 512
TEXT_CHARS = 900

TEMPERATURE = 0.0
TOP_P = 1.0
DO_SAMPLE = False

# Production should be False.
RESET_OUTPUT = False

# First pass should be fast and should not retry inline.
# Failed rows go to FAILED_FILE and can be retried later.
INLINE_RETRY = False

# Optional limit for testing. Set to None for full run.
TARGET_LIMIT = 100

print("Config loaded")
print("INPUT_FILE:", INPUT_FILE)
print("OUT_DIR:", OUT_DIR)
print("OUTPUT_FILE:", OUTPUT_FILE)
print("FAILED_FILE:", FAILED_FILE)


In [ ]:
# 3. Optional reset for testing only

if RESET_OUTPUT:
    for p in [OUTPUT_FILE, FAILED_FILE, FAILED_RETRY_FILE, FAILED_STILL_FILE, CHECKPOINT_FILE]:
        p.unlink(missing_ok=True)
    print("Reset output/checkpoint files.")
else:
    print("RESET_OUTPUT=False; existing output/checkpoint files will be preserved.")


In [ ]:
# 4. Output schema

RAG_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "english_summary",
        "legal_topic",
        "legal_question",
        "legal_rule",
        "court_holding",
        "factual_context",
        "english_legal_concepts",
        "search_keywords",
        "natural_language_queries",
    ],
    "properties": {
        "english_summary": {"type": "string", "maxLength": 220},
        "legal_topic": {"type": "string", "maxLength": 100},
        "legal_question": {"type": "string", "maxLength": 180},
        "legal_rule": {"type": "string", "maxLength": 220},
        "court_holding": {"type": "string", "maxLength": 180},
        "factual_context": {"type": "string", "maxLength": 180},
        "english_legal_concepts": {
            "type": "array",
            "items": {"type": "string", "maxLength": 60},
            "minItems": 0,
            "maxItems": 5,
        },
        "search_keywords": {
            "type": "array",
            "items": {"type": "string", "maxLength": 60},
            "minItems": 0,
            "maxItems": 6,
        },
        "natural_language_queries": {
            "type": "array",
            "items": {"type": "string", "maxLength": 140},
            "minItems": 0,
            "maxItems": 2,
        },
    },
}


In [ ]:
# 5. JSONL and checkpoint helpers

import json
from pathlib import Path
from typing import Iterator

def append_jsonl(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")
        f.flush()

def stream_jsonl(path: Path) -> Iterator[tuple[int, dict]]:
    with path.open("r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                yield line_idx, json.loads(line)
            except Exception as e:
                yield line_idx, {"_bad_json_line": line, "_line_error": str(e)}

def load_checkpoint(path: Path) -> set[int]:
    if not path.exists():
        return set()
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        return set(int(x) for x in data.get("processed_line_idxs", []))
    except Exception as e:
        print("Could not load checkpoint, starting fresh:", repr(e))
        return set()

def save_checkpoint(path: Path, processed: set[int]):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps({"processed_line_idxs": sorted(processed)}, ensure_ascii=False),
        encoding="utf-8",
    )
    tmp.replace(path)

def chunked(items, size: int):
    batch = []
    for x in items:
        batch.append(x)
        if len(batch) >= size:
            yield batch
            batch = []
    if batch:
        yield batch

print("Helper functions loaded.")


In [ ]:
# 6. Card text extraction

def get_first_present(card: dict, keys: list[str], default: str = "") -> str:
    for k in keys:
        v = card.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return default

def card_identifier(card: dict):
    for k in ["id", "card_id", "_id", "uuid", "source_id"]:
        if k in card:
            return card.get(k)
    return None

def extract_card_text(card: dict, max_chars: int = TEXT_CHARS) -> str:
    if "_bad_json_line" in card:
        return card.get("_bad_json_line", "")[:max_chars]

    candidates = [
        "text", "content", "paragraph", "body", "quote", "passage",
        "source_text", "card_text", "full_text", "raw_text",
    ]
    text = get_first_present(card, candidates)
    if not text:
        # Fall back to compact JSON so the model has some context.
        text = json.dumps(card, ensure_ascii=False)[:max_chars]
    return text[:max_chars]

def make_failure_obj(line_idx: int, card: dict, error: Exception | str, stage: str, raw_output: str | None = None):
    return {
        "line_idx": line_idx,
        "card_id": card_identifier(card),
        "card": card,
        "raw_output": raw_output,
        "error": str(error),
        "stage": stage,
    }


In [ ]:
# 7. Prompt

SYSTEM_INSTRUCTION = """You are a legal-data enrichment engine.
Return only valid JSON matching the requested schema.
Do not include markdown, explanations, citations, or extra keys.
Use concise English.
If information is not available, use an empty string or an empty array.
"""

def render_prompt(card: dict) -> str:
    text = extract_card_text(card, TEXT_CHARS)

    source_hint_parts = []
    for k in ["case_name", "court", "jurisdiction", "year", "citation", "source", "url"]:
        v = card.get(k)
        if v:
            source_hint_parts.append(f"{k}: {v}")
    source_hint = "\n".join(source_hint_parts)

    schema_hint = json.dumps(RAG_SCHEMA, ensure_ascii=False)

    return f"""{SYSTEM_INSTRUCTION}

JSON schema:
{schema_hint}

Source metadata:
{source_hint}

Card text:
{text}

Return one JSON object only:
"""


In [ ]:
# 8. JSON parsing and validation

import re
from jsonschema import validate

def extract_json_object(text: str) -> dict:
    if not isinstance(text, str):
        raise ValueError("Model output is not text")

    s = text.strip()

    # Remove common fences.
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s)
        s = re.sub(r"\s*```$", "", s)

    # Try direct parse first.
    try:
        return json.loads(s)
    except Exception:
        pass

    # Try extracting first object.
    start = s.find("{")
    end = s.rfind("}")
    if start >= 0 and end > start:
        candidate = s[start:end + 1]
        return json.loads(candidate)

    raise ValueError("No JSON object found in model output")

def normalize_enriched(obj: dict) -> dict:
    props = RAG_SCHEMA["properties"]
    out = {}

    for k, spec in props.items():
        if spec["type"] == "string":
            v = obj.get(k, "")
            if not isinstance(v, str):
                v = "" if v is None else str(v)
            max_len = spec.get("maxLength")
            out[k] = v[:max_len] if max_len else v
        elif spec["type"] == "array":
            v = obj.get(k, [])
            if not isinstance(v, list):
                v = []
            max_items = spec.get("maxItems", len(v))
            item_max = spec.get("items", {}).get("maxLength")
            arr = []
            for item in v[:max_items]:
                if not isinstance(item, str):
                    item = str(item)
                arr.append(item[:item_max] if item_max else item)
            out[k] = arr

    return out

def parse_model_output(text: str) -> dict:
    obj = extract_json_object(text)
    obj = normalize_enriched(obj)
    validate(instance=obj, schema=RAG_SCHEMA)
    return obj


In [ ]:
# 9. Safe Transformers engine loader

# This cell should not throw. If model loading fails, ENGINE_READY=False and the notebook stops safely.

ENGINE_READY = False
tokenizer = None
model = None

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print("Loading model with Transformers safe engine...")
    print("MODEL_ID:", MODEL_ID)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        use_fast=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
    )

    model.eval()
    ENGINE_READY = True
    print("Model loaded successfully with Transformers safe engine.")

except Exception as e:
    import traceback
    ENGINE_READY = False
    print("MODEL LOAD FAILED SAFELY. No exception was re-raised.")
    print("Reason:", repr(e))
    print(traceback.format_exc()[-6000:])
    print()
    print("Recommended next actions:")
    print("1. Check that this Colab runtime has enough GPU memory.")
    print("2. Check that the installed transformers version supports this model.")
    print("3. Try a smaller non-MoE model if this architecture is unsupported by Transformers.")


In [ ]:
# 10. Generation helpers

def generate_texts_transformers(prompts: list[str], max_new_tokens: int) -> list[str]:
    if not ENGINE_READY:
        return []

    import torch

    encoded = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    )

    # Move tensors to the first model device.
    first_device = next(model.parameters()).device
    encoded = {k: v.to(first_device) for k, v in encoded.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=DO_SAMPLE,
            temperature=None if not DO_SAMPLE else TEMPERATURE,
            top_p=None if not DO_SAMPLE else TOP_P,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    texts = []
    input_lens = encoded["input_ids"].shape[1]
    for seq in outputs:
        gen_ids = seq[input_lens:]
        texts.append(tokenizer.decode(gen_ids, skip_special_tokens=True))
    return texts

def generate_batch(prompts: list[str], max_new_tokens: int = MAX_NEW_TOKENS) -> list[str]:
    if ENGINE != "transformers_safe":
        print("Only transformers_safe is enabled in this fresh safe notebook.")
        return []
    return generate_texts_transformers(prompts, max_new_tokens=max_new_tokens)


In [ ]:
# 11. Smoke test

if not ENGINE_READY:
    print("Engine is not ready. Fix model loading before running the main pass.")
else:
    test_card = {
        "id": "smoke-test",
        "text": "The court held that the claimant failed to prove causation and dismissed the appeal."
    }
    prompt = render_prompt(test_card)
    raw = generate_batch([prompt], max_new_tokens=MAX_NEW_TOKENS)[0]
    print("RAW OUTPUT:")
    print(raw[:2000])

    try:
        enriched = parse_model_output(raw)
        print("PARSED JSON:")
        print(json.dumps(enriched, indent=2, ensure_ascii=False))
    except Exception as e:
        print("Smoke output did not parse. This will be captured as failure in main run.")
        print("Error:", repr(e))


In [ ]:
# 12. Main pass: process input, capture failures, continue

from time import time
from tqdm.auto import tqdm

if not ENGINE_READY:
    print("Engine is not ready; main pass not started.")
elif not INPUT_FILE.exists():
    print(f"Input file does not exist: {INPUT_FILE}")
    print("Update INPUT_FILE in the config cell and rerun.")
else:
    processed = load_checkpoint(CHECKPOINT_FILE)
    print("Already processed line count from checkpoint:", len(processed))

    rows_iter = ((line_idx, card) for line_idx, card in stream_jsonl(INPUT_FILE) if line_idx not in processed)

    processed_this_run = 0
    ok_this_run = 0
    failed_this_run = 0
    t0 = time()

    batch_buffer = []
    for line_idx, card in rows_iter:
        if TARGET_LIMIT is not None and processed_this_run >= TARGET_LIMIT:
            break

        batch_buffer.append((line_idx, card))

        if len(batch_buffer) < BATCH_SIZE:
            continue

        prompts = [render_prompt(card) for _, card in batch_buffer]

        try:
            raw_outputs = generate_batch(prompts, max_new_tokens=MAX_NEW_TOKENS)
        except Exception as e:
            raw_outputs = [None] * len(batch_buffer)
            for li, c in batch_buffer:
                append_jsonl(FAILED_FILE, make_failure_obj(li, c, e, "batch_generation_failed"))
                processed.add(li)
                failed_this_run += 1
                processed_this_run += 1
            save_checkpoint(CHECKPOINT_FILE, processed)
            batch_buffer = []
            continue

        for (li, c), raw in zip(batch_buffer, raw_outputs):
            try:
                enriched = parse_model_output(raw)
                append_jsonl(OUTPUT_FILE, {
                    "line_idx": li,
                    "card_id": card_identifier(c),
                    "card": c,
                    "enriched": enriched,
                    "raw_output": raw,
                    "status": "ok",
                    "engine": ENGINE,
                    "model_id": MODEL_ID,
                })
                ok_this_run += 1
            except Exception as e:
                append_jsonl(FAILED_FILE, make_failure_obj(li, c, e, "parse_or_validation_failed", raw))
                failed_this_run += 1

            # Mark both success and failure as processed so resume does not repeat failed rows.
            processed.add(li)
            processed_this_run += 1

        save_checkpoint(CHECKPOINT_FILE, processed)

        elapsed = max(time() - t0, 1e-6)
        print(
            f"processed={processed_this_run} ok={ok_this_run} failed={failed_this_run} "
            f"rate={processed_this_run/elapsed:.3f} rows/sec"
        )

        batch_buffer = []

    # Flush final partial batch.
    if batch_buffer and (TARGET_LIMIT is None or processed_this_run < TARGET_LIMIT):
        prompts = [render_prompt(card) for _, card in batch_buffer]
        try:
            raw_outputs = generate_batch(prompts, max_new_tokens=MAX_NEW_TOKENS)
        except Exception as e:
            raw_outputs = [None] * len(batch_buffer)
            for li, c in batch_buffer:
                append_jsonl(FAILED_FILE, make_failure_obj(li, c, e, "batch_generation_failed"))
                processed.add(li)
                failed_this_run += 1
                processed_this_run += 1
            save_checkpoint(CHECKPOINT_FILE, processed)
        else:
            for (li, c), raw in zip(batch_buffer, raw_outputs):
                try:
                    enriched = parse_model_output(raw)
                    append_jsonl(OUTPUT_FILE, {
                        "line_idx": li,
                        "card_id": card_identifier(c),
                        "card": c,
                        "enriched": enriched,
                        "raw_output": raw,
                        "status": "ok",
                        "engine": ENGINE,
                        "model_id": MODEL_ID,
                    })
                    ok_this_run += 1
                except Exception as e:
                    append_jsonl(FAILED_FILE, make_failure_obj(li, c, e, "parse_or_validation_failed", raw))
                    failed_this_run += 1
                processed.add(li)
                processed_this_run += 1
            save_checkpoint(CHECKPOINT_FILE, processed)

    elapsed = max(time() - t0, 1e-6)
    print("Main pass finished.")
    print(f"processed_this_run={processed_this_run}")
    print(f"ok_this_run={ok_this_run}")
    print(f"failed_this_run={failed_this_run}")
    print(f"average_rate={processed_this_run/elapsed:.3f} rows/sec")
    print("OUTPUT_FILE:", OUTPUT_FILE)
    print("FAILED_FILE:", FAILED_FILE)


In [ ]:
# 13. Retry only failed rows

def stream_failed_cards(path: Path):
    if not path.exists():
        print("No failed file found:", path)
        return
    with path.open("r", encoding="utf-8") as f:
        for row in f:
            obj = json.loads(row)
            yield obj["line_idx"], obj["card"], obj

if not ENGINE_READY:
    print("Engine is not ready; retry pass not started.")
elif not FAILED_FILE.exists():
    print("No failed rows to retry.")
else:
    failed_rows = list(stream_failed_cards(FAILED_FILE))
    print("Failed rows available for retry:", len(failed_rows))

    retry_limit = None  # set e.g. 100 for testing
    if retry_limit is not None:
        failed_rows = failed_rows[:retry_limit]

    recovered = 0
    still_failed = 0

    for batch in tqdm(list(chunked(failed_rows, BATCH_SIZE))):
        prompts = [render_prompt(card) for _, card, _ in batch]

        try:
            raw_outputs = generate_batch(prompts, max_new_tokens=RETRY_MAX_NEW_TOKENS)
        except Exception as e:
            for li, card, fail_obj in batch:
                append_jsonl(FAILED_STILL_FILE, {
                    "line_idx": li,
                    "card_id": card_identifier(card),
                    "card": card,
                    "raw_output": None,
                    "error": str(e),
                    "previous_error": fail_obj.get("error"),
                    "stage": "retry_batch_generation_failed",
                })
                still_failed += 1
            continue

        for (li, card, fail_obj), raw in zip(batch, raw_outputs):
            try:
                enriched = parse_model_output(raw)
                append_jsonl(FAILED_RETRY_FILE, {
                    "line_idx": li,
                    "card_id": card_identifier(card),
                    "card": card,
                    "enriched": enriched,
                    "raw_output": raw,
                    "status": "recovered",
                    "previous_error": fail_obj.get("error"),
                    "engine": ENGINE,
                    "model_id": MODEL_ID,
                })
                recovered += 1
            except Exception as e:
                append_jsonl(FAILED_STILL_FILE, {
                    "line_idx": li,
                    "card_id": card_identifier(card),
                    "card": card,
                    "raw_output": raw,
                    "error": str(e),
                    "previous_error": fail_obj.get("error"),
                    "stage": "retry_parse_or_validation_failed",
                })
                still_failed += 1

    print("Retry pass finished.")
    print("recovered:", recovered)
    print("still_failed:", still_failed)
    print("FAILED_RETRY_FILE:", FAILED_RETRY_FILE)
    print("FAILED_STILL_FILE:", FAILED_STILL_FILE)
